In [1]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from lightgbm import LGBMRegressor
import pandas as pd
import numpy as np


# Path to specific Walmart CSV file
CSV_PATH = r"D:\walmart_dataset\final_data_walmart.csv"

print(" Loading Walmart Dataset from CSV...")
df_raw = pd.read_csv(CSV_PATH)

df_raw = df_raw.drop(columns=[ "Date","Season","DayOfWeek","Month","WeekOfYear"], errors='ignore')
df_raw.columns = df_raw.columns.str.strip()

# Define domains based on city
source_cities = ['Houston', 'Philadelphia', 'Phoenix', 'San Jose', 'Jacksonville', 'Austin']
target_cities = ['New York', 'Los Angeles', 'Chicago']

print(" Applying Manual One-Hot Encoding...")

source_df = df_raw[df_raw['city'].isin(source_cities)].copy()
target_df = df_raw[df_raw['city'].isin(target_cities)].copy()
cols_to_encode = ["city", "Type", "weather_condition", "Store", "Dept"]

source_df = pd.get_dummies(source_df, columns=cols_to_encode)
target_df = pd.get_dummies(target_df, columns=cols_to_encode)
source_df, target_df = source_df.align(target_df, join='left', axis=1, fill_value=0)

TARGET = "Weekly_Sales"

X_source = source_df.drop(columns=[TARGET])
y_source = source_df[TARGET]


X_target = target_df.drop(columns=[TARGET])
y_target = target_df[TARGET]




print(" Splitting 10% of target data for training...")
# Split the target set: 10% goes to train, the remaining 90% stays for testing
X_target_train, X_target_test, y_target_train, y_target_test = train_test_split(
    X_target, y_target, train_size=0.1, random_state=42
)

X_test = pd.concat(
    [X_source, X_target_test],
    ignore_index=True
)

y_test = pd.concat(
    [y_source, y_target_test],
    ignore_index=True
)

print(f" Target samples added to train set (10%): {len(X_target_train)}")
print(f" Remaining Target samples for testing (90%): {len(X_target_test)}")



lgb_params = {
    'n_estimators': 300,
    'learning_rate': 0.05,
    'num_leaves': 62,
    'max_depth': 20,
    'verbose': -1,
    'n_jobs': 1
}
model = LGBMRegressor(**lgb_params)

print(" Training LightGBM on 10% target domain...")
model.fit(X_target_train, y_target_train)

print(" Predicting on the remaining 90% target test set...")
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"MAE : {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R²  : {r2:.4f}")

 Loading Walmart Dataset from CSV...
 Applying Manual One-Hot Encoding...
 Splitting 10% of target data for training...
 Target samples added to train set (10%): 21409
 Remaining Target samples for testing (90%): 192689
 Training LightGBM on 10% target domain...
 Predicting on the remaining 90% target test set...
MAE : 6751.34
RMSE: 13284.49
R²  : 0.6511


In [2]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from lightgbm import LGBMRegressor
import pandas as pd
import numpy as np


# Path to your specific Walmart CSV file
CSV_PATH = r"D:\walmart_dataset\final_data_walmart.csv"

print(" Loading Walmart Dataset from CSV...")
df_raw = pd.read_csv(CSV_PATH)

df_raw = df_raw.drop(columns=[ "Date","Season","DayOfWeek","Month","WeekOfYear"], errors='ignore')
df_raw.columns = df_raw.columns.str.strip()

source_weather = ['Clouds', 'Rain', 'Snow']
target_weather = ['Clear']

print(" Applying Manual One-Hot Encoding...")

source_df = df_raw[df_raw['weather_condition'].isin(source_weather)].copy()
target_df = df_raw[df_raw['weather_condition'].isin(target_weather)].copy()
cols_to_encode = ["city", "Type", "weather_condition", "Store", "Dept"]


source_df = pd.get_dummies(source_df, columns=cols_to_encode)
target_df = pd.get_dummies(target_df, columns=cols_to_encode)
source_df, target_df = source_df.align(target_df, join='left', axis=1, fill_value=0)


TARGET = "Weekly_Sales"




X_source = source_df.drop(columns=[TARGET])
y_source = source_df[TARGET]


X_target = target_df.drop(columns=[TARGET])
y_target = target_df[TARGET]




print(" Splitting 10% of target data for training...")
# Split the target set: 10% goes to train, the remaining 90% stays for testing
X_target_train, X_target_test, y_target_train, y_target_test = train_test_split(
    X_target, y_target, train_size=0.1, random_state=42
)

X_test = pd.concat(
    [X_source, X_target_test],
    ignore_index=True
)

y_test = pd.concat(
    [y_source, y_target_test],
    ignore_index=True
)

print(f" Target samples added to train set (10%): {len(X_target_train)}")
print(f" Remaining Target samples for testing (90%): {len(X_target_test)}")



lgb_params = {
    'n_estimators': 300,
    'learning_rate': 0.05,
    'num_leaves': 62,
    'max_depth': 20,
    'verbose': -1,
    'n_jobs': 1
}
model = LGBMRegressor(**lgb_params)

print(" Training LightGBM on 10% target domain...")
model.fit(X_target_train, y_target_train)

print(" Predicting on the remaining 90% target test set...")
# Predict ONLY on the 90% test portion to avoid data leakage
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)

 Loading Walmart Dataset from CSV...
 Applying Manual One-Hot Encoding...
 Splitting 10% of target data for training...
 Target samples added to train set (10%): 2847
 Remaining Target samples for testing (90%): 25629
 Training LightGBM on 10% target domain...
 Predicting on the remaining 90% target test set...
MAE: 7675.546482649708
RMSE: 12726.629866963707
R²: 0.7351852982659887


In [3]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from lightgbm import LGBMRegressor
import pandas as pd
import numpy as np


# Path to your specific Walmart CSV file
CSV_PATH = r"D:\walmart_dataset\final_data_walmart.csv"

print(" Loading Walmart Dataset from CSV...")
df_raw = pd.read_csv(CSV_PATH)

df_raw = df_raw.drop(columns=[ "Date","Season","DayOfWeek","Month","WeekOfYear"], errors='ignore')
df_raw.columns = df_raw.columns.str.strip()

# Define domains based on city
#source_cities = ['Houston', 'Philadelphia', 'Phoenix', 'San Jose', 'Jacksonville', 'Austin']

print(" Applying Manual One-Hot Encoding...")
cols_to_encode = ["city", "Type", "weather_condition",
                    "Store", "Dept"]
df_encoded = pd.get_dummies(df_raw, columns=cols_to_encode)
df_encoded = df_encoded.fillna(0)

source_df = df_encoded[df_encoded['Is_Christmas_Season'] == 0].copy()
target_df = df_encoded[df_encoded['Is_Christmas_Season'] == 1].copy()




TARGET = "Weekly_Sales"




X_source = source_df.drop(columns=[TARGET])
y_source = source_df[TARGET]


X_target = target_df.drop(columns=[TARGET])
y_target = target_df[TARGET]




print(" Splitting 10% of target data for training...")
# Split the target set: 10% goes to train, the remaining 90% stays for testing
X_target_train, X_target_test, y_target_train, y_target_test = train_test_split(
    X_target, y_target, train_size=0.1, random_state=42
)

X_test = pd.concat(
    [X_source, X_target_test],
    ignore_index=True
)

y_test = pd.concat(
    [y_source, y_target_test],
    ignore_index=True
)
print(f" Target samples added to train set (10%): {len(X_target_train)}")
print(f" Remaining Target samples for testing (90%): {len(X_target_test)}")



lgb_params = {
    'n_estimators': 300,
    'learning_rate': 0.05,
    'num_leaves': 62,
    'max_depth': 20,
    'verbose': -1,
    'n_jobs': 1
}
model = LGBMRegressor(**lgb_params)

print(" Training LightGBM on 10% target domain...")
model.fit(X_target_train, y_target_train)

print(" Predicting on the remaining 90% target test set...")
# Predict ONLY on the 90% test portion to avoid data leakage
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"MAE : {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R²  : {r2:.4f}")

 Loading Walmart Dataset from CSV...
 Applying Manual One-Hot Encoding...
 Splitting 10% of target data for training...
 Target samples added to train set (10%): 1783
 Remaining Target samples for testing (90%): 16051
 Training LightGBM on 10% target domain...
 Predicting on the remaining 90% target test set...
MAE : 15328.04
RMSE: 21966.15
R²  : 0.0622


In [4]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from lightgbm import LGBMRegressor
import pandas as pd
import numpy as np


# Path to your specific Walmart CSV file
CSV_PATH = r"D:\walmart_dataset\final_data_walmart.csv"

print(" Loading Walmart Dataset from CSV...")
df_raw = pd.read_csv(CSV_PATH)

df_raw = df_raw.drop(columns=[ "Date","Season","DayOfWeek","Month","WeekOfYear"], errors='ignore')
df_raw.columns = df_raw.columns.str.strip()

# Define domains based on city
#source_cities = ['Houston', 'Philadelphia', 'Phoenix', 'San Jose', 'Jacksonville', 'Austin']

print(" Applying Manual One-Hot Encoding...")
cols_to_encode = ["city", "Type", "weather_condition",
                    "Store", "Dept"]


source_df = df_raw[df_raw['Store'].between(1, 30)].copy()
target_df = df_raw[df_raw['Store'].between(31, 45)].copy()
source_df = pd.get_dummies(source_df, columns=cols_to_encode)
target_df = pd.get_dummies(target_df, columns=cols_to_encode)
source_df, target_df = source_df.align(target_df, join='left', axis=1, fill_value=0)




TARGET = "Weekly_Sales"




X_source = source_df.drop(columns=[TARGET])
y_source = source_df[TARGET]


X_target = target_df.drop(columns=[TARGET])
y_target = target_df[TARGET]




print(" Splitting 10% of target data for training...")
# Split the target set: 10% goes to train, the remaining 90% stays for testing
X_target_train, X_target_test, y_target_train, y_target_test = train_test_split(
    X_target, y_target, train_size=0.1, random_state=42
)

X_test = pd.concat(
    [X_source, X_target_test],
    ignore_index=True
)

y_test = pd.concat(
    [y_source, y_target_test],
    ignore_index=True
)
print(f" Target samples added to train set (10%): {len(X_target_train)}")
print(f" Remaining Target samples for testing (90%): {len(X_target_test)}")



lgb_params = {
    'n_estimators': 300,
    'learning_rate': 0.05,
    'num_leaves': 62,
    'max_depth': 20,
    'verbose': -1,
    'n_jobs': 1
}
model = LGBMRegressor(**lgb_params)

print(" Training LightGBM on 10% target domain...")
model.fit(X_target_train, y_target_train)

print(" Predicting on the remaining 90% target test set...")
# Predict ONLY on the 90% test portion to avoid data leakage
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"MAE : {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R²  : {r2:.4f}")

 Loading Walmart Dataset from CSV...
 Applying Manual One-Hot Encoding...
 Splitting 10% of target data for training...
 Target samples added to train set (10%): 12687
 Remaining Target samples for testing (90%): 114183
 Training LightGBM on 10% target domain...
 Predicting on the remaining 90% target test set...
MAE : 6213.31
RMSE: 12122.33
R²  : 0.7175


In [5]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from lightgbm import LGBMRegressor
import pandas as pd
import numpy as np


# Path to your specific Walmart CSV file
CSV_PATH = r"D:\walmart_dataset\final_data_walmart.csv"

print(" Loading Walmart Dataset from CSV...")
df_raw = pd.read_csv(CSV_PATH)

df_raw = df_raw.drop(columns=[ "Date","DayOfWeek","Month","WeekOfYear"], errors='ignore')
df_raw.columns = df_raw.columns.str.strip()

# Define domains based on city
#source_cities = ['Houston', 'Philadelphia', 'Phoenix', 'San Jose', 'Jacksonville', 'Austin']

print(" Applying Manual One-Hot Encoding...")

cols_to_encode = ["city", "Type", "weather_condition",
                    "Store", "Dept"]

source_df = df_raw[df_raw['Season'].isin([1, 2,4])].copy()
target_df = df_raw[df_raw['Season'].isin([3])].copy()
source_df = pd.get_dummies(source_df, columns=cols_to_encode)
target_df = pd.get_dummies(target_df, columns=cols_to_encode)
source_df, target_df = source_df.align(target_df, join='left', axis=1, fill_value=0)





TARGET = "Weekly_Sales"




X_source = source_df.drop(columns=[TARGET])
y_source = source_df[TARGET]


X_target = target_df.drop(columns=[TARGET])
y_target = target_df[TARGET]




print(" Splitting 10% of target data for training...")
# Split the target set: 10% goes to train, the remaining 90% stays for testing
X_target_train, X_target_test, y_target_train, y_target_test = train_test_split(
    X_target, y_target, train_size=0.1, random_state=42
)

X_test = pd.concat(
    [X_source, X_target_test],
    ignore_index=True
)

y_test = pd.concat(
    [y_source, y_target_test],
    ignore_index=True
)
print(f" Target samples added to train set (10%): {len(X_target_train)}")
print(f" Remaining Target samples for testing (90%): {len(X_target_test)}")



lgb_params = {
    'n_estimators': 300,
    'learning_rate': 0.05,
    'num_leaves': 62,
    'max_depth': 20,
    'verbose': -1,
    'n_jobs': 1
}
model = LGBMRegressor(**lgb_params)

print(" Training LightGBM on 10% target domain...")
model.fit(X_target_train, y_target_train)

print(" Predicting on the remaining 90% target test set...")
# Predict ONLY on the 90% test portion to avoid data leakage
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"MAE : {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R²  : {r2:.4f}")

 Loading Walmart Dataset from CSV...
 Applying Manual One-Hot Encoding...
 Splitting 10% of target data for training...
 Target samples added to train set (10%): 11640
 Remaining Target samples for testing (90%): 104767
 Training LightGBM on 10% target domain...
 Predicting on the remaining 90% target test set...
MAE : 4742.58
RMSE: 9552.96
R²  : 0.8236


In [6]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from lightgbm import LGBMRegressor
import pandas as pd
import numpy as np


# Path to your specific Walmart CSV file
CSV_PATH = r"D:\walmart_dataset\final_data_walmart.csv"

print(" Loading Walmart Dataset from CSV...")
df_raw = pd.read_csv(CSV_PATH)

df_raw = df_raw.drop(columns=[ "Date","DayOfWeek","Month","WeekOfYear"], errors='ignore')
df_raw.columns = df_raw.columns.str.strip()

# Define domains based on city
#source_cities = ['Houston', 'Philadelphia', 'Phoenix', 'San Jose', 'Jacksonville', 'Austin']

print(" Applying Manual One-Hot Encoding...")

cols_to_encode = ["city", "Type", "weather_condition",
                    "Store", "Dept"]

source_df = df_raw[df_raw['Type'].isin(['C'])].copy()
target_df = df_raw[df_raw['Type'].isin(['A','B'])].copy()
source_df = pd.get_dummies(source_df, columns=cols_to_encode)
target_df = pd.get_dummies(target_df, columns=cols_to_encode)
source_df, target_df = source_df.align(target_df, join='left', axis=1, fill_value=0)

cols_to_encode = ["city", "Type", "weather_condition", "Store", "Dept"]




TARGET = "Weekly_Sales"




X_source = source_df.drop(columns=[TARGET])
y_source = source_df[TARGET]


X_target = target_df.drop(columns=[TARGET])
y_target = target_df[TARGET]




print(" Splitting 10% of target data for training...")
# Split the target set: 10% goes to train, the remaining 90% stays for testing
X_target_train, X_target_test, y_target_train, y_target_test = train_test_split(
    X_target, y_target, train_size=0.1, random_state=42
)

X_test = pd.concat(
    [X_source, X_target_test],
    ignore_index=True
)

y_test = pd.concat(
    [y_source, y_target_test],
    ignore_index=True
)
print(f" Target samples added to train set (10%): {len(X_target_train)}")
print(f" Remaining Target samples for testing (90%): {len(X_target_test)}")



lgb_params = {
    'n_estimators': 300,
    'learning_rate': 0.05,
    'num_leaves': 62,
    'max_depth': 20,
    'verbose': -1,
    'n_jobs': 1
}
model = LGBMRegressor(**lgb_params)

print(" Training LightGBM on 10% target domain...")
model.fit(X_target_train, y_target_train)

print(" Predicting on the remaining 90% target test set...")
# Predict ONLY on the 90% test portion to avoid data leakage
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"MAE : {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R²  : {r2:.4f}")

 Loading Walmart Dataset from CSV...
 Applying Manual One-Hot Encoding...
 Splitting 10% of target data for training...
 Target samples added to train set (10%): 37625
 Remaining Target samples for testing (90%): 338629
 Training LightGBM on 10% target domain...
 Predicting on the remaining 90% target test set...
MAE : 4109.72
RMSE: 7720.79
R²  : 0.8841


In [7]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from lightgbm import LGBMRegressor
import pandas as pd
import numpy as np


# Path to your specific Walmart CSV file
CSV_PATH = r"D:\walmart_dataset\final_data_walmart.csv"

print(" Loading Walmart Dataset from CSV...")
df_raw = pd.read_csv(CSV_PATH)

df_raw = df_raw.drop(columns=[ "Date","Season","DayOfWeek","Month","WeekOfYear"], errors='ignore')
df_raw.columns = df_raw.columns.str.strip()

# Define domains based on city
#source_cities = ['Houston', 'Philadelphia', 'Phoenix', 'San Jose', 'Jacksonville', 'Austin']

print(" Applying Manual One-Hot Encoding...")
cols_to_encode = ["city", "Type", "weather_condition",
                    "Store", "Dept"]
df_encoded = pd.get_dummies(df_raw, columns=cols_to_encode)
df_encoded = df_encoded.fillna(0)

source_df = df_encoded[df_encoded['IsHoliday'] == 0].copy().reset_index(drop=True)
target_df = df_encoded[df_encoded['IsHoliday'] == 1].copy().reset_index(drop=True)





TARGET = "Weekly_Sales"




X_source = source_df.drop(columns=[TARGET])
y_source = source_df[TARGET]


X_target = target_df.drop(columns=[TARGET])
y_target = target_df[TARGET]




print(" Splitting 10% of target data for training...")
# Split the target set: 10% goes to train, the remaining 90% stays for testing
X_target_train, X_target_test, y_target_train, y_target_test = train_test_split(
    X_target, y_target, train_size=0.1, random_state=42
)

X_test = pd.concat(
    [X_source, X_target_test],
    ignore_index=True
)

y_test = pd.concat(
    [y_source, y_target_test],
    ignore_index=True
)
print(f" Target samples added to train set (10%): {len(X_target_train)}")
print(f" Remaining Target samples for testing (90%): {len(X_target_test)}")



lgb_params = {
    'n_estimators': 300,
    'learning_rate': 0.05,
    'num_leaves': 62,
    'max_depth': 20,
    'verbose': -1,
    'n_jobs': 1
}
model = LGBMRegressor(**lgb_params)

print(" Training LightGBM on 10% target domain...")
model.fit(X_target_train, y_target_train)

print(" Predicting on the remaining 90% target test set...")
# Predict ONLY on the 90% test portion to avoid data leakage
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"MAE : {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R²  : {r2:.4f}")

 Loading Walmart Dataset from CSV...
 Applying Manual One-Hot Encoding...
 Splitting 10% of target data for training...
 Target samples added to train set (10%): 2929
 Remaining Target samples for testing (90%): 26364
 Training LightGBM on 10% target domain...
 Predicting on the remaining 90% target test set...
MAE : 7296.25
RMSE: 12551.73
R²  : 0.6947
